# Demo de Flujo: Orquestador Multi-Agente Especializado

Este notebook demuestra el flujo de delegación del sistema importando la lógica directamente desde el código fuente.

**Requisitos:**
- API Key de Google Gemini
- API Key de Pinecone

**Flujo:**
1. Setup: Clone e instalación
2. Configuración del proyecto
3. Configuración de API keys
4. Validación de Gemini y Pinecone
5. Carga de documentos e ingesta
6. Ejecución del grafo con visualización del flujo

## 1. Setup: Clone e instalación

Clonamos el repositorio, cambiamos al directorio del proyecto y instalamos las dependencias.

In [ ]:
import os
import sys

# Clone solo si no existe (re-ejecutable)
if not os.path.exists("orquestador_multi_agente"):
    !git clone https://github.com/santiotero/orquestador_multi_agente.git

os.chdir("orquestador_multi_agente")
sys.path.insert(0, os.getcwd())
print(f"Directorio de trabajo: {os.getcwd()}")

# Instalación de dependencias
!pip install -q langgraph langgraph-checkpoint-sqlite langchain langchain-core \
    langchain-classic langchain-community langchain-google-genai \
    langchain-pinecone langchain-text-splitters pinecone rank-bm25 \
    pypdf pydantic python-dotenv aiofiles aiosqlite tiktoken ipywidgets

## 2. Configuración del proyecto

Cargamos la configuración central desde `config/config.json`.

In [ ]:
from src.schema.models import cargar_configuracion

cfg = cargar_configuracion()
print(f"Config cargada: modelo={cfg.model_name}, índice={cfg.index_name}, namespace={cfg.namespace}")

## 3. Configuración de API Keys

Configuramos las API keys necesarias de forma segura.

In [ ]:
import getpass
import os

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Introduce tu Gemini API Key: ")

if "PINECONE_API_KEY" not in os.environ:
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Introduce tu Pinecone API Key: ")

print("API keys configuradas correctamente.")

## 4. Validación de Gemini

Validamos que la API key de Gemini sea válida con una invocación de prueba.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

def validar_gemini(api_key, model_name):
    """Valida que la API key de Gemini sea válida con una invocación de prueba."""
    try:
        llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0.0,
            max_output_tokens=10,
            api_key=api_key,
        )
        respuesta = llm.invoke("responde con una palabra: OK")
        if respuesta and respuesta.content:
            print(f"API key de Gemini válida. Modelo: {model_name}")
            return True
        else:
            print(f"ERROR: La API key de Gemini no devolvió respuesta.")
            print(f"Causa probable: Key incorrecta, expirada o sin permisos.")
            print(f"Acción: Ve a aistudio.google.com → API Keys → Copia tu key y vuelve a ingresarla.")
            raise SystemExit(1)
    except Exception as e:
        error_str = str(e).lower()
        if "api key" in error_str or "permission" in error_str:
            print(f"ERROR: API key de Gemini inválida.")
            print(f"Causa probable: Key incorrecta, expirada o sin permisos.")
            print(f"Acción: Ve a aistudio.google.com → API Keys → Copia tu key y vuelve a ingresarla.")
        elif "model" in error_str or "not found" in error_str:
            print(f"ERROR: El modelo '{model_name}' no está disponible.")
            print(f"Causa probable: Modelo incorrecto o no habilitado en tu cuenta.")
            print(f"Acción: Verifica el nombre del modelo en config/config.json o usa 'gemini-2.0-flash'.")
        elif "quota" in error_str or "rate" in error_str:
            print(f"ERROR: Cuota de API excedida.")
            print(f"Causa probable: Límite de requests por minuto o día alcanzado.")
            print(f"Acción: Espera unos minutos o verifica tu cuota en aistudio.google.com.")
        else:
            print(f"ERROR: No se pudo validar la API key de Gemini.")
            print(f"Causa probable: Error de red, key incorrecta o problema con el modelo.")
            print(f"Acción: Verifica tu conexión a internet y que la API key sea correcta.")
        print(f"Detalle técnico: {e}")
        raise SystemExit(1)

validar_gemini(os.environ["GEMINI_API_KEY"], cfg.model_name)

## 5. Validación de Pinecone

Validamos que la API key, índice y namespace de Pinecone existan y estén configurados correctamente.

In [ ]:
from pinecone import Pinecone

def validar_pinecone(api_key, index_name, namespace):
    """Valida que la API key, índice y namespace existan en Pinecone."""
    try:
        pc = Pinecone(api_key=api_key)
        indices = [idx.name for idx in pc.list_indexes()]
        print(f"API key válida. Índices disponibles: {indices}")
    except Exception as e:
        print(f"ERROR: API key de Pinecone inválida.")
        print(f"Causa probable: Key incorrecta, expirada o sin permisos.")
        print(f"Acción: Ve a pinecone.io → API Keys → Copia tu key y vuelve a ingresarla.")
        print(f"Detalle técnico: {e}")
        raise SystemExit(1)

    if index_name not in indices:
        print(f"ERROR: El índice '{index_name}' no existe en tu cuenta de Pinecone.")
        print(f"Causa probable: Nunca se creó o fue eliminado.")
        print(f"Acción: Crea un índice serverless en AWS us-east-1 con 1536 dimensiones y métrica cosine,")
        print(f"        o actualiza config/config.json con el nombre correcto del índice.")
        print(f"Índices disponibles: {indices}")
        raise SystemExit(1)

    index = pc.Index(index_name)
    stats = index.describe_index_stats()

    if namespace not in stats.namespaces:
        print(f"ERROR: El namespace '{namespace}' no existe en el índice '{index_name}'.")
        print(f"Causa probable: Nunca se ejecutó la ingesta de datos.")
        print(f"Acción: Ejecuta la ingesta de datos (sección 7 del notebook) para crear el namespace automáticamente.")
        raise SystemExit(1)

    if stats.namespaces[namespace].vector_count == 0:
        print(f"ERROR: El namespace '{namespace}' existe pero no tiene vectores.")
        print(f"Causa probable: Ingesta fallida o incompleta.")
        print(f"Acción: Ejecuta la ingesta de datos (sección 7 del notebook) para cargar los documentos.")
        raise SystemExit(1)

    print(f"Pinecone configurado correctamente:")
    print(f"  - Índice: {index_name}")
    print(f"  - Namespace: {namespace}")
    print(f"  - Vectores: {stats.namespaces[namespace].vector_count}")
    return True

validar_pinecone(os.environ["PINECONE_API_KEY"], cfg.index_name, cfg.namespace)

## 6. Carga de documentos

Si no hay documentos en `ingest/data/`, se solicitan subir archivos.

In [ ]:
import os
import shutil
from google.colab import files

data_dir = "ingest/data"
os.makedirs(data_dir, exist_ok=True)

documentos = [f for f in os.listdir(data_dir) if f.endswith(('.pdf', '.txt', '.md'))]

if not documentos:
    print("ERROR: No se encontraron documentos (PDF, TXT, MD) en ingest/data/.")
    print("Causa probable: Carpeta vacía o archivos no subidos.")
    print("Acción: Sube al menos un documento PDF, TXT o MD en la siguiente celda.")
    print("\nSelecciona los archivos de tu proyecto:")

    uploaded = files.upload()

    if not uploaded:
        print("ERROR: No se subieron archivos.")
        print("Acción: Sube al menos un documento para continuar.")
        raise SystemExit(1)

    for filename in uploaded.keys():
        if filename.endswith(('.pdf', '.txt', '.md')):
            shutil.move(filename, os.path.join(data_dir, filename))
            print(f"  Movido: {filename} -> ingest/data/")
        else:
            print(f"  ADVERTENCIA: '{filename}' tiene un formato no soportado. Usa solo .pdf, .txt o .md")

    documentos = [f for f in os.listdir(data_dir) if f.endswith(('.pdf', '.txt', '.md'))]
    if not documentos:
        print("ERROR: No se subieron documentos válidos.")
        print("Acción: Sube al menos un archivo .pdf, .txt o .md para continuar.")
        raise SystemExit(1)

print(f"Documentos encontrados: {documentos}")

## 7. Ingesta de datos

Ejecutamos la ingesta de datos en Pinecone y BM25.

In [ ]:
from src.vectorstore.rag_system import RAGSystem
from ingest.ingest import main as ingest_main

rag = RAGSystem(cfg, os.environ["PINECONE_API_KEY"], os.environ["GEMINI_API_KEY"])

if rag.has_data() and rag.has_bm25():
    print("Datos ya existen en Pinecone y BM25. Omitiendo ingesta.")
else:
    print("Ejecutando ingesta...")
    try:
        ingest_main()
        print("Ingesta completada exitosamente.")
    except Exception as e:
        print(f"ERROR: La ingesta falló.")
        print(f"Causa probable: Error en Pinecone, embeddings o documentos inválidos.")
        print(f"Acción: Verifica tu API key de Pinecone, que el índice exista y que los documentos sean válidos.")
        print(f"Detalle técnico: {e}")
        raise SystemExit(1)

## 8. Diagrama del grafo

Topología del orquestador multi-agente:

```mermaid
graph TD
    subgraph Grafo["Orquestador Multi-Agente"]
        START([START]) --> supervisor
        supervisor -->|next_agent| investigador
        supervisor -->|next_agent| analista
        supervisor -->|next_agent| sintesis
        investigador -->|contribucion| supervisor
        analista -->|contribucion| supervisor
        sintesis --> END([END])
    end

    subgraph Estado["Estado Compartido"]
        messages["messages"]
        contribuciones["contribuciones"]
        intentos["intentos_correccion"]
        instruccion["instruccion_actual"]
        next_agent["next_agent"]
    end

    supervisor -.lee.-> Estado
    investigador -.escribe.-> contribuciones
    analista -.escribe.-> contribuciones
    sintesis -.lee.-> Estado
```

## 9. Pregunta del usuario

Escribe tu pregunta para el orquestador.

In [ ]:
import ipywidgets as widgets

pregunta_widget = widgets.Textarea(
    value="Una API lee 400 millones de filas y escribe 20 millones al mes en DynamoDB ($0,125 y $1,25 por millón). ¿Cuánto cuestan las operaciones?",
    placeholder="Escribe tu pregunta...",
    layout=widgets.Layout(width="100%", height="100px")
)

display(pregunta_widget)

## 10. Ejecución del grafo

Construimos el grafo y ejecutamos el flujo de delegación paso a paso.

In [ ]:
from src.graph import construir_grafo
from src.providers.gemini import GeminiClient
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_core.messages import HumanMessage
from uuid import uuid4

cliente = GeminiClient(cfg)

async def ejecutar_demo():
    async with AsyncSqliteSaver.from_conn_string(":memory:") as saver:
        grafo = construir_grafo(cliente, cfg, saver)
        config = {
            "configurable": {"thread_id": f"demo-{uuid4().hex[:8]}"},
            "recursion_limit": cfg.recursion_limit,
        }
        async for updates in grafo.astream(
            {"messages": [HumanMessage(pregunta_widget.value)], "intentos_correccion": 0},
            config,
            stream_mode="updates",
        ):
            for nodo, datos in updates.items():
                if nodo == "supervisor":
                    print(f"[SUPERVISOR] -> {datos['next_agent']}: {datos['instruccion_actual'][:100]}...")
                elif nodo in ("investigador", "analista"):
                    for contribucion in datos.get("contribuciones", []):
                        print(f"[{contribucion['agente'].upper()}] {contribucion['contenido'][:150]}...")
                elif nodo == "sintesis":
                    print(f"[SINTESIS] {datos['messages'][-1].content[:200]}...")
        estado = await grafo.aget_state(config)
        return estado

estado_final = await ejecutar_demo()

print("\n=== CONTRIBUCIONES FINALES ===")
for contribucion in estado_final.values.get("contribuciones", []):
    print(f"\n[{contribucion['agente'].upper()}]")
    print(contribucion['contenido'])

## 11. Traza de la sesión

Mostramos el archivo `.jsonl` generado durante la ejecución.

In [ ]:
import json
from pathlib import Path

trace_files = list(Path("trace").glob("trace_*.jsonl"))
if trace_files:
    with open(trace_files[-1]) as f:
        for line in f:
            evento = json.loads(line)
            print(f"{evento['tipo']}: {evento.get('contenido', evento.get('decision', ''))[:100]}")
else:
    print("No se encontraron archivos de traza.")